# Python Bootcamp — Foundation Phase Summary
## Days 1–20 

This notebook demonstrates core Python skills built across the Foundation phase,  
applied to a simulated manufacturing production dataset.

**Skills demonstrated:**
- Variables, data types, control flow, functions
- Object-oriented programming
- CSV ingestion and data validation  
- Pandas: filtering, groupby, pivot tables, aggregation
- Matplotlib: bar charts, line charts, trend analysis
- End-to-end pipeline thinking

**Dataset:** `sample_data.csv` — 240 simulated production records  
30 days × 4 machines × 2 shifts | `random.seed(42)`

## 1. Setup & Configuration

In [ ]:
# Foundation Summary — Day 20
# Python Bootcamp 

import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
import random

# Constants
TARGET_RATE    = 1.8     # Units per minute
OEE_THRESHOLD  = 0.85   # World class threshold
MIN_ACCEPTABLE = 0.70   # Minimum acceptable OEE
RANDOM_SEED    = 42

random.seed(RANDOM_SEED)

print(f"Notebook initialized : {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print(f"OEE Threshold        : {OEE_THRESHOLD*100:.0f}%")
print(f"Target Rate          : {TARGET_RATE} units/min")

## 2. Data Ingestion & Validation

Loading production data with column validation and type enforcement.  
Demonstrates: CSV ingestion, error handling, type conversion.

In [ ]:
def load_and_validate(filepath):
    """
    Load production CSV with validation.

    Args:
        filepath (str): Path to CSV file

    Returns:
        pd.DataFrame or None
    """
    try:
        df = pd.read_csv(filepath)
    except FileNotFoundError:
        print(f"[ERROR] File not found: {filepath}")
        return None

    required = ["date", "machine", "shift",
                "actual_run", "planned", "units",
                "good_units", "defect_code"]

    missing = [c for c in required if c not in df.columns]
    if missing:
        print(f"[ERROR] Missing columns: {missing}")
        return None

    # Type enforcement
    df["date"]       = pd.to_datetime(df["date"])
    df["actual_run"] = pd.to_numeric(df["actual_run"], errors="coerce")
    df["planned"]    = pd.to_numeric(df["planned"],    errors="coerce")
    df["units"]      = pd.to_numeric(df["units"],      errors="coerce")
    df["good_units"] = pd.to_numeric(df["good_units"], errors="coerce")

    df.dropna(subset=["actual_run", "planned", "units", "good_units"],
              inplace=True)
    df = df[df["planned"] > 0]

    print(f"[OK] Loaded {df.shape[0]} records × {df.shape[1]} columns")
    print(f"     Date range: {df['date'].min().date()} "
          f"to {df['date'].max().date()}")
    print(f"     Machines  : {sorted(df['machine'].unique())}")
    return df

df = load_and_validate("sample_data.csv")

## 3. OEE Transformation

Calculating Availability, Performance, Quality, and OEE  
using vectorized Pandas operations.  
Demonstrates: vectorization, lambda classification, derived columns.

In [ ]:
def transform(df, target_rate=TARGET_RATE, threshold=OEE_THRESHOLD):
    """Calculate OEE metrics and classify each record."""
    df = df.copy()
    df["availability"] = df["actual_run"] / df["planned"]
    df["performance"]  = df["units"] / (df["planned"] * target_rate)
    df["quality"]      = df["good_units"] / df["units"]
    df["oee"]          = (df["availability"] *
                          df["performance"] *
                          df["quality"])
    df["status"] = df["oee"].apply(
        lambda x: "World Class"    if x >= threshold
             else "Acceptable"     if x >= MIN_ACCEPTABLE
             else "Below Threshold"
    )
    print(f"[OK] OEE calculated")
    print(f"     Range  : {df['oee'].min()*100:.1f}% "
          f"— {df['oee'].max()*100:.1f}%")
    print(f"     Average: {df['oee'].mean()*100:.1f}%")
    return df

df = transform(df)
df.head(3)

## 4. Fleet Analysis

Grouped aggregation by machine center with performance ranking.  
Demonstrates: groupby, agg, rank, sort, apply.

In [ ]:
fleet = df.groupby("machine").agg(
    avg_oee      = ("oee", "mean"),
    std_oee      = ("oee", "std"),
    min_oee      = ("oee", "min"),
    max_oee      = ("oee", "max"),
    total_units  = ("units", "sum"),
    record_count = ("oee", "count")
).round(3).reset_index()

fleet["rank"] = fleet["avg_oee"].rank(ascending=False).astype(int)
fleet["status"] = fleet["avg_oee"].apply(
    lambda x: "World Class"    if x >= OEE_THRESHOLD
         else "Acceptable"     if x >= MIN_ACCEPTABLE
         else "Below Threshold"
)
fleet = fleet.sort_values("rank")

print("Fleet Performance Summary:")
print(fleet[["rank", "machine", "avg_oee",
             "std_oee", "min_oee", "total_units", "status"]]
      .to_string(index=False))

## 5. Shift Analysis

Pivot table showing OEE by machine and shift.  
Demonstrates: pivot_table, multi-dimensional groupby.

In [ ]:
pivot = df.pivot_table(
    values  = "oee",
    index   = "machine",
    columns = "shift",
    aggfunc = "mean"
).round(3)

print("OEE by Machine and Shift:")
print(pivot)

# Visualize
pivot.plot(kind="bar", figsize=(10, 5), edgecolor="black")
plt.axhline(y=OEE_THRESHOLD, color="red", linestyle="--",
            linewidth=1.5, label=f"World Class ({OEE_THRESHOLD*100:.0f}%)")
plt.title("OEE by Machine and Shift", fontsize=14, fontweight="bold")
plt.ylabel("Average OEE")
plt.ylim(0, 1.0)
plt.xticks(rotation=15, ha="right")
plt.legend()
plt.tight_layout()
plt.savefig("foundation_pivot_chart.png", dpi=150)
plt.show()

## 6. Defect Analysis

Pareto view of defect code frequency.  
Demonstrates: value_counts, filtering, horizontal bar chart.

In [ ]:
defects = df[df["defect_code"] != "NONE"]["defect_code"].value_counts()

plt.figure(figsize=(10, 5))
bars = plt.barh(defects.index, defects.values,
                color="tomato", edgecolor="black")
for bar, val in zip(bars, defects.values):
    plt.text(bar.get_width() + 0.5,
             bar.get_y() + bar.get_height() / 2,
             str(val), va="center", fontsize=10)

plt.title("Defect Code Frequency — Pareto View",
          fontsize=14, fontweight="bold")
plt.xlabel("Occurrence Count")
plt.ylabel("Defect Code")
plt.tight_layout()
plt.savefig("foundation_defect_pareto.png", dpi=150)
plt.show()

print(f"\nTop defect code: {defects.index[0]} "
      f"({defects.values[0]} occurrences)")

## 7. Trend Analysis

7-day rolling OEE average for CNC Mill #4.  
Demonstrates: datetime filtering, rolling window, multi-line chart.

In [ ]:
cnc = df[df["machine"] == "CNC Mill #4"].copy()
cnc_daily = cnc.groupby("date")["oee"].mean().reset_index()
cnc_daily.columns = ["date", "daily_oee"]
cnc_daily["rolling_7d"] = cnc_daily["daily_oee"].rolling(
    window=7, min_periods=1).mean()

plt.figure(figsize=(12, 5))
plt.plot(cnc_daily["date"], cnc_daily["daily_oee"],
         color="lightsteelblue", linewidth=1.5,
         marker="o", markersize=3, label="Daily OEE")
plt.plot(cnc_daily["date"], cnc_daily["rolling_7d"],
         color="steelblue", linewidth=2.5,
         label="7-Day Rolling Avg")
plt.axhline(y=OEE_THRESHOLD, color="red", linestyle="--",
            linewidth=1.5, label="World Class")
plt.axhline(y=MIN_ACCEPTABLE, color="orange", linestyle="--",
            linewidth=1.5, label="Min Acceptable")
plt.title("CNC Mill #4 — OEE Trend with 7-Day Rolling Average",
          fontsize=14, fontweight="bold")
plt.xlabel("Date")
plt.ylabel("OEE")
plt.ylim(0, 1.0)
plt.xticks(rotation=45, ha="right")
plt.legend()
plt.tight_layout()
plt.savefig("foundation_trend.png", dpi=150)
plt.show()

## 8. Foundation Phase Summary

### Skills Demonstrated

| Day | Topic | Applied As |
|---|---|---|
| 1–3 | Variables, strings, conditionals | OEE classification logic |
| 4–5 | Lists, loops, functions | Data processing utilities |
| 6–7 | Dictionaries, error handling | Safe ingestion with validation |
| 8–9 | Modules, CSV ingestion | `oee_utils.py`, `sample_data.csv` |
| 10–11 | OOP, inheritance | `MachineRecord` class hierarchy |
| 12 | Comprehensions | Vectorized list operations |
| 13–15 | Jupyter, Pandas, Matplotlib | Notebook pipeline and charts |
| 16 | Advanced Pandas | Merge, pivot, cleaning |
| 17 | Advanced OOP | Abstract base classes |
| 18 | Data pipeline | 5-stage ingest→report pipeline |
| 19 | Pandas deep dive | Rolling averages, crosstab |
| 20 | Foundation summary | This notebook |

### Key Metrics from This Analysis

In [ ]:
report_time = datetime.now().strftime("%Y-%m-%d %H:%M")

print("=" * 54)
print(f"  FOUNDATION PHASE SUMMARY — {report_time}")
print("=" * 54)
print(f"  Dataset        : {len(df)} records")
print(f"  Date range     : {df['date'].min().date()} "
      f"to {df['date'].max().date()}")
print(f"  Fleet avg OEE  : {df['oee'].mean()*100:.1f}%")
print(f"  World Class    : "
      f"{(df['status']=='World Class').sum()} records")
print(f"  Below threshold: "
      f"{(df['status']=='Below Threshold').sum()} records")
print(f"  Top machine    : "
      f"{fleet.iloc[0]['machine']} "
      f"({fleet.iloc[0]['avg_oee']*100:.1f}%)")
print(f"  Needs attention: "
      f"{fleet.iloc[-1]['machine']} "
      f"({fleet.iloc[-1]['avg_oee']*100:.1f}%)")
print("=" * 54)
print("\n20-day Foundation Phase complete.")
print("GitHub commits: 20")
print("Next phase    : Building Blocks (Days 21-55)")